In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("ag_news")

# Split into train and test
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

**Reload Embeddings**

In [3]:
import numpy as np

X_train = np.load("/kaggle/input/newsclassifier-model/X_train.npy")
X_test = np.load("/kaggle/input/newsclassifier-model/X_test.npy")
y_train = np.load("/kaggle/input/newsclassifier-model/y_train.npy")
y_test = np.load("/kaggle/input/newsclassifier-model/y_test.npy")

print("Embeddings Reloaded")

Embeddings Reloaded


**SVM Reload**

In [4]:
import joblib

# Load model
svm_model = joblib.load("/kaggle/input/newsclassifier-model/svm_model.pkl")

print("Model Loaded")

Model Loaded


**RandomForest Reload**

In [5]:
import joblib

# Load model
rf_model = joblib.load("/kaggle/input/newsclassifier-model/rf_model.pkl")

print("Model Loaded")

Model Loaded


**FFNN Reload**

In [7]:
import torch
import torch.nn as nn

# Define model class again
class FFNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(FFNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(out)

# Initialize and load model
input_size = X_train.shape[1]  # Usually 384 if using MiniLM
model_ffnn = FFNN(input_size=input_size, hidden_size=128, num_classes=4)
model_ffnn.load_state_dict(torch.load("/kaggle/input/newsclassifier-model/ffnn_model.pth"))
model_ffnn.eval()

print("Model Loaded")

Model Loaded


/tmp/ipykernel_31/525237651.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_ffnn.load_state_dict(torch.load("/kaggle/input/newsclassifier-model/ffnn_model.pth"))


**RNN**

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convert data to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Optional: reshape embeddings to simulate sequence format for RNN
# Example: 384 → (sequence_length=24, input_size=16)
X_train_tensor = X_train_tensor.view(-1, 24, 16)
X_test_tensor = X_test_tensor.view(-1, 24, 16)

# Create DataLoaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64)

# Define the RNN model 
class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(RNNClassifier, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.rnn(x)        # <- this returns both output and hidden state
        out = out[:, -1, :]         # get the last time step
        out = self.fc(out)
        return out

# Hyperparameters
input_size = 16         # Based on reshaped dimensions
hidden_size = 128
num_layers = 1
num_classes = 4         # AG News
num_epochs = 5
lr = 1e-3

# Initialize model
model_rnn = RNNClassifier(input_size, hidden_size, num_layers, num_classes)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=lr)

# Training loop
for epoch in range(num_epochs):
    model_rnn.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model_rnn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluation
model_rnn.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model_rnn(batch_x)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

print(f"RNN Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Loss: 0.3498
Epoch 2, Loss: 0.3160
Epoch 3, Loss: 0.4093
Epoch 4, Loss: 0.4073
Epoch 5, Loss: 0.3834
RNN Accuracy: 87.83%


**Save RNN Model**

In [13]:
# Save the trained RNN model
torch.save(model_rnn.state_dict(), "rnn_classifier.pth")
print("Model saved as rnn_classifier.pth")

Model saved as rnn_classifier.pth


In [14]:
from sentence_transformers import SentenceTransformer

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

2025-05-07 11:30:12.155799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746617412.363722      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746617412.426257      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
import torch

def predict_news_category(news_text, model, model_type="sklearn"):
    labels = ["World", "Sports", "Business", "Sci/Tech"]
    
    # Embed the input text
    embedded = embedder.encode([news_text])  # shape: (1, 384)

    if model_type == "sklearn":
        prediction = model.predict(embedded)[0]
        return labels[prediction]
    
    elif model_type in ["ffnn", "rnn"]:
        model.eval()
        with torch.no_grad():
            tensor_input = torch.tensor(embedded, dtype=torch.float32)  # shape: (1, 384)

            if model_type == "rnn":
                # Reshape for RNN: (batch_size, seq_len, input_size)
                tensor_input = tensor_input.view(1, 24, 16)  # e.g., (1, 24, 16)
            
            # FFNN takes (1, 384), RNN takes (1, 24, 16)
            logits = model(tensor_input)
            _, predicted = torch.max(logits, 1)
            return labels[predicted.item()]
    
    else:
        raise ValueError("Invalid model_type. Use 'sklearn', 'ffnn', or 'rnn'.")

# Example usage
custom_news = "NASA launches a new satellite to explore Jupiter."

print("Predicted Category (SVM):", predict_news_category(custom_news, svm_model, model_type="sklearn"))
print("Predicted Category (Random Forest):", predict_news_category(custom_news, rf_model, model_type="sklearn"))
print("Predicted Category (FFNN):", predict_news_category(custom_news, model_ffnn, model_type="ffnn"))
print("Predicted Category (RNN):", predict_news_category(custom_news, model_rnn, model_type="rnn"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (SVM): Sci/Tech


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (Random Forest): Sci/Tech


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (FFNN): Sci/Tech


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (RNN): Sci/Tech


In [23]:
import torch

def predict_news_category(news_text, model, model_type="sklearn"):
    labels = ["World", "Sports", "Business", "Sci/Tech"]
    
    # Embed the input text
    embedded = embedder.encode([news_text])  # shape: (1, 384)

    if model_type == "sklearn":
        prediction = model.predict(embedded)[0]
        return labels[prediction]
    
    elif model_type in ["ffnn", "rnn"]:
        model.eval()
        with torch.no_grad():
            tensor_input = torch.tensor(embedded, dtype=torch.float32)  # shape: (1, 384)

            if model_type == "rnn":
                # Reshape for RNN: (batch_size, seq_len, input_size)
                tensor_input = tensor_input.view(1, 24, 16)  # e.g., (1, 24, 16)
            
            # FFNN takes (1, 384), RNN takes (1, 24, 16)
            logits = model(tensor_input)
            _, predicted = torch.max(logits, 1)
            return labels[predicted.item()]
    
    else:
        raise ValueError("Invalid model_type. Use 'sklearn', 'ffnn', or 'rnn'.")

# Example usage
custom_news = input("Enter a news:")

print("Predicted Category (SVM):", predict_news_category(custom_news, svm_model, model_type="sklearn"))
print("Predicted Category (Random Forest):", predict_news_category(custom_news, rf_model, model_type="sklearn"))
print("Predicted Category (FFNN):", predict_news_category(custom_news, model_ffnn, model_type="ffnn"))
print("Predicted Category (RNN):", predict_news_category(custom_news, model_rnn, model_type="rnn"))

Enter a news: pakistan win by 3 runs


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (SVM): World


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (Random Forest): Sports


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (FFNN): Sports


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted Category (RNN): Sports


**Check Percentage**

In [29]:
from collections import Counter
import pandas as pd

# Assuming you're using the train split
train_data = dataset["train"]
labels = train_data["label"]

# Count each label
label_counts = Counter(labels)

# Total number of samples
total = len(labels)

# Convert to percentage
percentages = {label: f"{(count / total) * 100:.2f}%" for label, count in label_counts.items()}

# Optional: Show as DataFrame with label names
label_names = ["World", "Sports", "Business", "Sci/Tech"]
percent_df = pd.DataFrame({
    "Label Index": list(label_counts.keys()),
    "Label Name": [label_names[i] for i in label_counts.keys()],
    "Count": list(label_counts.values()),
    "Percentage": list(percentages.values())
})

print(percent_df)

   Label Index Label Name  Count Percentage
0            2   Business  30000     25.00%
1            3   Sci/Tech  30000     25.00%
2            1     Sports  30000     25.00%
3            0      World  30000     25.00%
